In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Imports

In [2]:
import warnings

import numpy as np
import pandas as pd

from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from statsmodels.stats.outliers_influence import variance_inflation_factor

Define paths and constants

In [3]:
PROCESSED_DATA_DIR = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/"
EXP_1_OUTPUT_DIR = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/exp_1/"

MAIN_TRAIN_DATA_PATH = PROCESSED_DATA_DIR + "hcc_train_full.csv"
MAIN_TEST_DATA_PATH = PROCESSED_DATA_DIR + "hcc_test_full.csv"

TRAIN_PATH = EXP_1_OUTPUT_DIR + "hcc_exp1_train_selected.csv"
TEST_PATH = EXP_1_OUTPUT_DIR + "hcc_exp1_test_selected.csv"
SELECTED_FEATURES_PATH = EXP_1_OUTPUT_DIR + "hcc_exp1_selected_features.csv"

ID_COLUMNS = ["PATIENT_ID", "SAMPLE_ID"]
DURATION_COL = "OS_MONTHS"
EVENT_COL = "OS_EVENT"
TARGET_COLUMNS = [DURATION_COL, EVENT_COL]

COMPONENTS = 50
VIF_THRESHOLD = 10
MIN_VARIANCE = 0.001

In [4]:
import os

os.makedirs(EXP_1_OUTPUT_DIR, exist_ok=True)

Load Train and test data

In [5]:
train_df = pd.read_csv(MAIN_TRAIN_DATA_PATH)
test_df = pd.read_csv(MAIN_TEST_DATA_PATH)

train_df.head()

,PATIENT_ID,SAMPLE_ID,SAMPLE_COVERAGE,TUMOR_PURITY,MSI_SCORE,TUMOR_SIZE,TMB_NONSYNONYMOUS,DIAGNOSIS_AGE,BMI,BILIARY_SCORE,...,NEOADJUVANT_SYSTEMIC_THERAPY_No,NEOADJUVANT_SYSTEMIC_THERAPY_Yes,ADJUVANT_SYSTEMIC_THERAPY_,ADJUVANT_SYSTEMIC_THERAPY_No,ADJUVANT_SYSTEMIC_THERAPY_Yes,PALLIATIVE_SYSTEMIC_THERPAY_,PALLIATIVE_SYSTEMIC_THERPAY_No,PALLIATIVE_SYSTEMIC_THERPAY_Yes,OS_MONTHS,OS_EVENT
0,P-0011104,P-0011104-T01-IM5,886.0,60.0,0.00,6.4,5.872318,53.0,19.2,0.716,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,82.296,0
1,P-0001336,P-0001336-T01-IM3,755.0,30.0,0.16,6.4,4.436621,61.0,40.7,0.763,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,51.324,1
2,P-0064295,P-0064295-T01-IM7,635.0,70.0,0.50,5.4,4.101736,74.0,24.4,0.532,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,6.600,1
3,P-0049360,P-0049360-T01-IM6,537.0,30.0,0.12,6.4,4.323490,61.0,37.2,0.737,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,24.480,0
4,P-0010532,P-0010532-T01-IM5,1056.0,30.0,0.16,3.8,2.936159,63.0,20.9,0.763,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,14.592,1


In [6]:
test_df.head()

,PATIENT_ID,SAMPLE_ID,SAMPLE_COVERAGE,TUMOR_PURITY,MSI_SCORE,TUMOR_SIZE,TMB_NONSYNONYMOUS,DIAGNOSIS_AGE,BMI,BILIARY_SCORE,...,NEOADJUVANT_SYSTEMIC_THERAPY_No,NEOADJUVANT_SYSTEMIC_THERAPY_Yes,ADJUVANT_SYSTEMIC_THERAPY_,ADJUVANT_SYSTEMIC_THERAPY_No,ADJUVANT_SYSTEMIC_THERAPY_Yes,PALLIATIVE_SYSTEMIC_THERPAY_,PALLIATIVE_SYSTEMIC_THERPAY_No,PALLIATIVE_SYSTEMIC_THERPAY_Yes,OS_MONTHS,OS_EVENT
0,P-0016909,P-0016909-T01-IM6,913.0,70.0,0.00,6.8,0.000000,59.0,24.4,0.763,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,28.980,1
1,P-0032234,P-0032234-T01-IM6,560.0,60.0,0.33,13.7,4.323490,76.0,26.0,0.951,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,5.952,1
2,P-0045819,P-0045819-T01-IM6,592.0,90.0,1.61,4.5,6.052887,59.0,29.9,0.763,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,9.984,0
3,P-0052368,P-0052368-T02-IM6,642.0,40.0,0.05,6.7,4.323490,63.0,27.7,0.868,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,23.652,1
4,P-0019203,P-0019203-T01-IM6,634.0,50.0,1.29,4.3,1.729396,57.0,21.6,0.763,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,36.696,0


In [7]:
train_df.shape, test_df.shape

((1085, 1196), (272, 1196))

In [8]:
print("Train OS_EVENT distribution:")
print(train_df[EVENT_COL].value_counts())

print("\nTest OS_EVENT distribution:")
print(test_df[EVENT_COL].value_counts())

Train OS_EVENT distribution:
OS_EVENT
1    641
0    444
Name: count, dtype: int64

Test OS_EVENT distribution:
OS_EVENT
1    161
0    111
Name: count, dtype: int64


Separate features and target

In [9]:
feature_columns = [
    col for col in train_df.columns
    if col not in ID_COLUMNS + TARGET_COLUMNS
]

X_train = train_df[feature_columns]
X_test = test_df[feature_columns]
y_train = train_df[EVENT_COL]

X_train.shape, X_test.shape, y_train.shape

((1085, 1192), (272, 1192), (1085,))

Remove constant features

In [10]:
constant_filter = VarianceThreshold(threshold=0.0)

X_train_nonconstant = pd.DataFrame(
    constant_filter.fit_transform(X_train),
    columns=X_train.columns[constant_filter.get_support()],
    index=X_train.index,
)

X_test_nonconstant = X_test[X_train_nonconstant.columns]

removed_constant_features = sorted(set(feature_columns) - set(X_train_nonconstant.columns))

print("Original feature count:", len(feature_columns))
print("Non-constant feature count:", X_train_nonconstant.shape[1])
print("Removed constant features:", len(removed_constant_features))

Original feature count: 1192
Non-constant feature count: 1020
Removed constant features: 172


Select 50 best features using Select-K Best

In [11]:
k_best = min(COMPONENTS, X_train_nonconstant.shape[1])

selector = SelectKBest(score_func=f_classif, k=k_best)
selector.fit(X_train_nonconstant, y_train)

selected_features = X_train_nonconstant.columns[selector.get_support()].tolist()

print("Selected feature count:", len(selected_features))
selected_features[:20]

Selected feature count: 50


['SAMPLE_COVERAGE',
 'TUMOR_SIZE',
 'MUT_ALB',
 'MUT_ARID1A',
 'MUT_ARID1B',
 'MUT_UPF1',
 'MUT_WHSC1',
 'MUT_ZFHX3',
 'CNA_CDKN2A',
 'CNA_KMT2B',
 'CNA_CDKN2Ap16INK4A',
 'SAMPLE_TYPE_Metastasis',
 'SAMPLE_TYPE_Primary',
 'METASTATIC_SITE_Not Applicable',
 'PRIMARY_SITE_Liver',
 'GENE_PANEL_IMPACT341',
 'GENE_PANEL_IMPACT410',
 'GENE_PANEL_IMPACT468',
 'GENE_PANEL_IMPACT505',
 'PRED_CATEGORY_Biliary class']

Remove high-VIF features

In [12]:
def calculate_vif(df: pd.DataFrame) -> pd.DataFrame:
    records = []
    values = df.astype(float).values

    for idx, col in enumerate(df.columns):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            try:
                score = variance_inflation_factor(values, idx)
            except Exception:
                score = np.inf

        records.append({"Feature": col, "VIF": score})

    return pd.DataFrame(records)

In [13]:
X_vif = X_train_nonconstant[selected_features].copy()
dropped_vif_features = []

while X_vif.shape[1] > 1:
    vif_df = calculate_vif(X_vif)
    max_vif_row = vif_df.sort_values("VIF", ascending=False).iloc[0]

    if (not np.isfinite(max_vif_row["VIF"])) or (max_vif_row["VIF"] > VIF_THRESHOLD):
        dropped_vif_features.append(max_vif_row["Feature"])
        X_vif = X_vif.drop(columns=[max_vif_row["Feature"]])
    else:
        break

vif_selected_features = X_vif.columns.tolist()

print("Features removed due to high VIF:", len(dropped_vif_features))
print("Features remaining after VIF filtering:", len(vif_selected_features))
dropped_vif_features[:20]

Features removed due to high VIF: 18
Features remaining after VIF filtering: 32


['GENE_PANEL_IMPACT341',
 'C_N_No',
 'ADJUVANT_SYSTEMIC_THERAPY_',
 'ADJUVANT_SYSTEMIC_THERAPY_No',
 'PALLIATIVE_SYSTEMIC_THERPAY_',
 'PALLIATIVE_SYSTEMIC_THERPAY_Yes',
 'CURATIVE_INTENT_SURGERY_No',
 'METASTASIS_No',
 'SAMPLE_TYPE_Primary',
 'MAJOR_LIVER_RESECTION_',
 'METASTATIC_SITE_Not Applicable',
 'LVI_',
 'RESECTION_MARGIN_',
 'HISTO_SUBTYPE_',
 'CNA_CDKN2A',
 'MULTIFOCAL_DISEASE_No',
 'ETHNICITY_Non-Spanish; Non-Hispanic',
 'CURATIVE_INTENT_SURGERY_Yes']

Remove low-variance features within survival groups

In [14]:
events = train_df[EVENT_COL].astype(bool)
low_variance_features = []

for column in vif_selected_features:
    event_variance = train_df.loc[events, column].var()
    censored_variance = train_df.loc[~events, column].var()

    if event_variance < MIN_VARIANCE or censored_variance < MIN_VARIANCE:
        low_variance_features.append(column)

final_features = [
    col for col in vif_selected_features
    if col not in low_variance_features
]

print("Low-variance features removed:", len(low_variance_features))
print("Final selected feature count:", len(final_features))
final_features

Low-variance features removed: 2
Final selected feature count: 30


['SAMPLE_COVERAGE',
 'TUMOR_SIZE',
 'MUT_ALB',
 'MUT_ARID1A',
 'MUT_ARID1B',
 'MUT_UPF1',
 'MUT_WHSC1',
 'MUT_ZFHX3',
 'CNA_CDKN2Ap16INK4A',
 'SAMPLE_TYPE_Metastasis',
 'PRIMARY_SITE_Liver',
 'GENE_PANEL_IMPACT410',
 'GENE_PANEL_IMPACT468',
 'GENE_PANEL_IMPACT505',
 'PRED_CATEGORY_Biliary class',
 'ETHNICITY_Unknown whether Spanish or not',
 'MULTIFOCAL_DISEASE_Yes',
 'LVI_No',
 'PNI_',
 'PNI_No',
 'P_N_',
 'P_N_Negative',
 'C_N_',
 'C_N_Yes',
 'METASTASIS_Yes',
 'MAJOR_LIVER_RESECTION_No',
 'RESECTION_MARGIN_R0',
 'NEOADJUVANT_SYSTEMIC_THERAPY_',
 'ADJUVANT_SYSTEMIC_THERAPY_Yes',
 'PALLIATIVE_SYSTEMIC_THERPAY_No']

Build final train and test datasets

In [15]:
train_df_final = train_df[final_features + TARGET_COLUMNS].copy().reset_index(drop=True)
test_df_final = test_df[final_features + TARGET_COLUMNS].copy().reset_index(drop=True)

train_df_final.shape, test_df_final.shape

((1085, 32), (272, 32))

In [16]:
print("Missing values in final train data:", int(train_df_final.isna().sum().sum()))
print("Missing values in final test data:", int(test_df_final.isna().sum().sum()))

Missing values in final train data: 0
Missing values in final test data: 0


In [17]:
train_df_final.head()

,SAMPLE_COVERAGE,TUMOR_SIZE,MUT_ALB,MUT_ARID1A,MUT_ARID1B,MUT_UPF1,MUT_WHSC1,MUT_ZFHX3,CNA_CDKN2Ap16INK4A,SAMPLE_TYPE_Metastasis,...,C_N_,C_N_Yes,METASTASIS_Yes,MAJOR_LIVER_RESECTION_No,RESECTION_MARGIN_R0,NEOADJUVANT_SYSTEMIC_THERAPY_,ADJUVANT_SYSTEMIC_THERAPY_Yes,PALLIATIVE_SYSTEMIC_THERPAY_No,OS_MONTHS,OS_EVENT
0,886.0,6.4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,82.296,0
1,755.0,6.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,51.324,1
2,635.0,5.4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,6.600,1
3,537.0,6.4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,24.480,0
4,1056.0,3.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,14.592,1


In [18]:
test_df_final.shape

(272, 32)

Save selected datasets

In [19]:
train_df_final.to_csv(TRAIN_PATH, index=False)
test_df_final.to_csv(TEST_PATH, index=False)

selected_features_df = pd.DataFrame({"Feature": final_features})
selected_features_df.to_csv(SELECTED_FEATURES_PATH, index=False)

TRAIN_PATH, TEST_PATH, SELECTED_FEATURES_PATH

('/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/exp_1/hcc_exp1_train_selected.csv',
 '/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/exp_1/hcc_exp1_test_selected.csv',
 '/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/exp_1/hcc_exp1_selected_features.csv')

Final check

In [20]:
print("Final train shape:", train_df_final.shape)
print("Final test shape:", test_df_final.shape)

print("\nFinal train OS_EVENT distribution:")
print(train_df_final[EVENT_COL].value_counts())

print("\nFinal test OS_EVENT distribution:")
print(test_df_final[EVENT_COL].value_counts())

Final train shape: (1085, 32)
Final test shape: (272, 32)

Final train OS_EVENT distribution:
OS_EVENT
1    641
0    444
Name: count, dtype: int64

Final test OS_EVENT distribution:
OS_EVENT
1    161
0    111
Name: count, dtype: int64


\This notebook performs feature selection for the first baseline experiment. Its role is to reduce the very wide survival dataset into a smaller, cleaner, more stable feature set that can be used in CoxPH and DeepSurv training.
The notebook reads:

hcc_train_full.csv
hcc_test_full.csv

These are the outputs of the preprocessing notebook.
The first technical step is to separate identifiers and targets from candidate predictor columns. The following columns are excluded from feature selection:

PATIENT_ID
SAMPLE_ID
OS_MONTHS
OS_EVENT

This is done for a very important reason. PATIENT_ID and SAMPLE_ID are preserved for traceability and future XAI, but they are not valid predictive variables. OS_MONTHS and OS_EVENT are the target columns, so including them in feature selection would leak the answer into the model.

The notebook then works only with the remaining candidate features.
The first filtering step removes constant features. These are columns that do not vary across the training set and therefore carry no predictive information. In a multi-omic dataset like this one, constant columns are common, especially after one-hot encoding and sparse mutation feature construction.

The second step uses SelectKBest with f_classif to keep the top 50 features most associated with the event label OS_EVENT. This step acts as a first-pass ranking filter and narrows the search space from more than a thousand candidate features to a smaller core group.

The next step applies VIF-based filtering. VIF, or Variance Inflation Factor, is used to identify redundant features that carry overlapping information. This is especially important for CoxPH, because strongly collinear features can make the model unstable and difficult to interpret. In this notebook, high-VIF features are removed iteratively until the selected feature set becomes more stable.

The last selection step removes features with extremely low variance within survival groups. This is a more careful version of the logic used in the original project. Because our current dataset includes sparse binary mutation features, a large variance threshold would wrongly remove many biologically meaningful mutation variables. For that reason, the threshold is set very low:
MIN_VARIANCE = 0.001

That makes the filtering conservative rather than destructive.
After all three selection steps are complete, the notebook builds final selected train and test datasets containing:

the final selected features
OS_MONTHS
OS_EVENT

The identifiers are not carried into these selected modeling files because the immediate purpose of these outputs is model fitting. The IDs remain available in the earlier full train-test files and can be reattached later when needed for interpretation or reporting.
This notebook generates:

hcc_exp1_train_selected.csv
hcc_exp1_test_selected.csv
hcc_exp1_selected_features.csv

Conceptually, this notebook is the bridge between raw wide survival data and model-ready compact data. It reduces dimensionality, removes redundancy, and prepares a stable input space for the first CoxPH and DeepSurv experiment.